In [ ]:
import os
import pandas as pd
import scanpy as sc
import liana as li
import matplotlib.pyplot as plt
from plotnine import labs, theme, element_text
import warnings

warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# DIRECTORY SETUP
# ---------------------------------------------------------
DATA_DIR = './data/'
OUTPUT_LIANA = './results/tables/liana/'
OUTPUT_FIGS = './results/figures/liana/'

os.makedirs(OUTPUT_LIANA, exist_ok=True)
os.makedirs(OUTPUT_FIGS, exist_ok=True)

# ---------------------------------------------------------
# CORE FUNCTIONS
# ---------------------------------------------------------
def run_liana_consensus(adata, sample_key='Sample', groupby='Type'):
    """
    Run LIANA rank aggregate (consensus) across samples.
    """
    print(f"[*] Running LIANA consensus for {adata.n_obs} cells...")
    li.mt.rank_aggregate.by_sample(
        adata,
        sample_key=sample_key,
        groupby=groupby,
        resource_name='consensus',
        expr_prop=0.1,
        min_cells=5,
        n_perms=100,
        use_raw=False,
        verbose=False,
        inplace=True
    )
    return adata

def plot_interaction_network(adata, condition_name, output_dir):
    """
    Generate and save a circular network plot of cell-cell interactions.
    """
    print(f"[*] Generating interaction network for {condition_name}...")
    p = li.pl.circle_plot(
        adata,
        groupby='Type',
        score_key='magnitude_rank',
        inverse_score=True,
        pivot_mode='counts',
        filter_fun=lambda x: x['specificity_rank'] <= 0.05,
        figure_size=(8, 8)
    )

    plt.suptitle(f"{condition_name} Interaction Network", fontsize=16, fontweight='bold', y=0.90)

    file_path = os.path.join(output_dir, f"Network_{condition_name}.pdf")
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  -> Saved to {file_path}")

def plot_source_target_dotplot(adata, source_cell, condition_name, output_dir, top_n=20):
    """
    Generate and save a dotplot of top interactions originating from a specific source cell.
    """
    print(f"[*] Generating {source_cell} dotplot for {condition_name}...")
    target_labels = ['T cells', 'Malignant cells', 'TAMs', 'TECs', 'B cells', 'CAFs']

    p = li.pl.dotplot(
        adata=adata,
        colour='magnitude_rank',
        size='specificity_rank',
        inverse_size=True,
        inverse_colour=True,
        source_labels=[source_cell],
        target_labels=target_labels,
        top_n=top_n,
        orderby='magnitude_rank',
        orderby_ascending=True,
        figure_size=(8, 7)
    )

    p = p + labs(title=f"{source_cell} interactions ({condition_name})") + \
        theme(
            plot_title=element_text(size=16, weight='bold', vjust=0),
            plot_margin=0.04
        )

    file_path = os.path.join(output_dir, f"Dotplot_{condition_name}_{source_cell}.pdf")
    p.save(file_path, dpi=300, height=8, width=9, verbose=False)
    print(f"  -> Saved to {file_path}")

def calculate_interaction_degree(adata, condition_name, output_dir=None):
    """
    Calculate and export interaction degrees (Senders/Receivers) into a summary table.
    """
    df_res = adata.uns['liana_res']
    df_filtered = df_res[df_res['specificity_rank'] <= 0.05].copy()

    sender_counts = df_filtered['source'].value_counts()
    receiver_counts = df_filtered['target'].value_counts()

    summary_df = pd.DataFrame({
        f'{condition_name}_Senders': sender_counts,
        f'{condition_name}_Receivers': receiver_counts
    }).fillna(0).astype(int)

    summary_df[f'{condition_name}_Total'] = summary_df[f'{condition_name}_Senders'] + summary_df[f'{condition_name}_Receivers']
    summary_df = summary_df.sort_values(by=f'{condition_name}_Total', ascending=False)
    summary_df.index.name = 'Cell_Types'

    if output_dir:
        file_path = os.path.join(output_dir, f"Interaction_Degree_{condition_name}.csv")
        summary_df.to_csv(file_path)

    return summary_df

# =========================================================
# EXECUTION PIPELINE
# =========================================================
print("=== 1. LOAD DATA ===")
adata = sc.read_h5ad(os.path.join(DATA_DIR, 'GSE151530_processed.h5ad'))

# Run LIANA on full dataset and save complete results
adata = run_liana_consensus(adata)
adata.uns['liana_res'].to_csv(os.path.join(OUTPUT_LIANA, 'LIANA_Global_Results.csv'), index=False)
adata_liana_path = os.path.join(DATA_DIR, 'GSE151530_liana.h5ad')
adata.write_h5ad(adata_liana_path, compression='gzip')
print(f"[*] Full AnnData with LIANA results saved to: {adata_liana_path}")

print("\n=== 2. SPLIT & RUN LIANA FOR DISEASE CONTEXTS ===")
# Split data into HCC and iCCA
adata_hcc = adata[adata.obs["Disease"].str.startswith("H", na=False)].copy()
adata_icca = adata[adata.obs["Disease"].str.startswith("iC", na=False)].copy()

# Run LIANA separately
adata_hcc = run_liana_consensus(adata_hcc)
adata_icca = run_liana_consensus(adata_icca)

print("\n=== 3. NETWORK VISUALIZATION ===")
plot_interaction_network(adata_hcc, condition_name="HCC", output_dir=OUTPUT_FIGS)
plot_interaction_network(adata_icca, condition_name="iCCA", output_dir=OUTPUT_FIGS)

print("\n=== 4. EXPORT INTERACTION SUMMARIES ===")
df_degree_hcc = calculate_interaction_degree(adata_hcc, condition_name="HCC", output_dir=OUTPUT_LIANA)
df_degree_icca = calculate_interaction_degree(adata_icca, condition_name="iCCA", output_dir=OUTPUT_LIANA)

# Merge into a single Supplementary Table
supp_table = pd.merge(df_degree_hcc, df_degree_icca, on='Cell_Types', how='outer').fillna(0).astype(int)
supp_table.to_csv(os.path.join(OUTPUT_LIANA, "Supplementary_Table_Interaction_Degrees.csv"))
print(f"[*] Combined Interaction Degree Table saved.")

print("\n=== 5. SPECIFIC AXIS VISUALIZATION (CAFs) ===")
plot_source_target_dotplot(adata_hcc, source_cell="CAFs", condition_name="HCC", output_dir=OUTPUT_FIGS)
plot_source_target_dotplot(adata_icca, source_cell="CAFs", condition_name="iCCA", output_dir=OUTPUT_FIGS)

print("\n[*] PIPELINE COMPLETED SUCCESSFULLY.")